In [ ]:
# MUSCLE / Precision Resistance Training — Stage 1 (ONE CELL, ROBUST)
from google.colab import drive
import subprocess, sys, json, shutil, traceback
from datetime import datetime
from pathlib import Path

drive.mount('/content/drive')

REPO_URL = 'https://github.com/kimtk94/IS_Analysis_V3.git'
BRANCH = 'main'
ROOT = Path('/content/MUSCLE_IS_Analysis_V3')
DRIVE_ROOT = Path('/content/drive/MyDrive/MASTER_DEGREE/MUSCLE')
OUTDIR = DRIVE_ROOT / 'results/stage1_gwas_audit'
LOGDIR = DRIVE_ROOT / 'logs'
OUTDIR.mkdir(parents=True, exist_ok=True)
LOGDIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
LOGFILE = LOGDIR / f'MUSCLE_STAGE1_{stamp}.log'
log_lines = []

def emit(x=''):
    x = str(x)
    print(x)
    log_lines.append(x)

def run(cmd, cwd=None):
    cmd = [str(x) for x in cmd]
    emit('$ ' + ' '.join(cmd))
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        emit(p.stdout.rstrip())
    if p.stderr:
        emit(p.stderr.rstrip())
    emit(f'[returncode={p.returncode}]')
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {" ".join(cmd)}')
    return p

try:
    emit('===== MUSCLE STAGE 1 START =====')
    emit(f'Python: {sys.version}')
    emit(f'Drive root: {DRIVE_ROOT}')

    # Always use a fresh clone so another Colab session/repo cannot contaminate this run.
    if ROOT.exists():
        shutil.rmtree(ROOT)

    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, ROOT])

    required = [
        ROOT / 'scripts/muscle_stage1_gwas_audit.py',
        ROOT / 'tests/test_muscle_stage1_gwas_audit.py',
        ROOT / 'config/muscle/stage1_seed_loci.tsv',
    ]
    for p in required:
        if not p.exists():
            raise FileNotFoundError(f'Missing repo file: {p}')
        emit(f'FOUND {p.relative_to(ROOT)}')

    run([sys.executable, '-m', 'py_compile', ROOT / 'scripts/muscle_stage1_gwas_audit.py'])

    # Run test file directly; avoids Colab/site-packages package-name collisions around "tests".
    run([sys.executable, ROOT / 'tests/test_muscle_stage1_gwas_audit.py'], cwd=ROOT)

    run([
        sys.executable,
        ROOT / 'scripts/muscle_stage1_gwas_audit.py',
        '--input', ROOT / 'config/muscle/stage1_seed_loci.tsv',
        '--outdir', OUTDIR,
    ])

    expected = [
        OUTDIR / 'MUSCLE_STAGE1_GWAS_AUDIT.tsv',
        OUTDIR / 'MUSCLE_STAGE1_SUMMARY.json',
        OUTDIR / 'MUSCLE_STAGE1_REPORT.md',
    ]
    for p in expected:
        if not p.exists() or p.stat().st_size == 0:
            raise RuntimeError(f'Expected output missing/empty: {p}')
        emit(f'OUTPUT {p.name} | {p.stat().st_size:,} bytes')

    summary = json.loads((OUTDIR / 'MUSCLE_STAGE1_SUMMARY.json').read_text())
    emit('')
    emit('===== MUSCLE STAGE 1 COMPLETE =====')
    emit(json.dumps(summary, indent=2, ensure_ascii=False))
    emit(f'Saved to: {OUTDIR}')

except Exception:
    emit('')
    emit('===== MUSCLE STAGE 1 ERROR =====')
    emit(traceback.format_exc())
    raise

finally:
    LOGFILE.write_text('\n'.join(log_lines) + '\n', encoding='utf-8')
    print(f'\nRun log saved to: {LOGFILE}')
